# LigandMPNN Inverse Folding

This notebook demonstrates how to use `run_ligandmpnn_sample` to design protein sequences conditioned on both backbone structure and nearby ligand atoms, and `run_ligandmpnn_score` to score sequence-structure compatibility in the same context. LigandMPNN extends ProteinMPNN by incorporating small molecule geometry into sequence design, enabling the creation of binding pocket sequences that are compatible with a specific ligand conformation. When no ligand atoms are present in the input structure, LigandMPNN behaves similarly to ProteinMPNN.


In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("ligandmpnn")
display_overview("ligandmpnn")
display_docs_section("ligandmpnn", "Background")

# LigandMPNN

LigandMPNN is an inverse folding model that designs protein sequences conditioned on a backbone structure and its molecular context -- including bound ligands, metal ions, nucleic acids, and non-standard residues. It extends ProteinMPNN's message-passing neural network architecture to incorporate atomic-level information from the non-protein environment surrounding the design target.

- **Tool keys**: `ligandmpnn-sample`, `ligandmpnn-score`
- **Input**: Protein structures (PDB/CIF) with optional chain and position constraints
- **Output**: Designed sequences and sequence-structure scores
- **Execution**: GPU required

## Background

[Inverse folding](https://en.wikipedia.org/wiki/Protein_design#Inverse_folding) solves the "reverse" protein design problem: given a desired 3D backbone structure, what amino acid sequence will fold into that structure? This is the complement of structure prediction (sequence -> structure).

ProteinMPNN (Dauparas et al., 2022) pioneered the [message-passing neural network](https://en.wikipedia.org/wiki/Graph_neural_network) approach for inverse folding, achieving recovery rates (~50% native sequence identity) far exceeding previous physics-based methods. However, ProteinMPNN only considers the protein backbone and ignores non-protein molecules.

LigandMPNN extends this by encoding the atomic coordinates and chemical identities of:
- **Small-molecule ligands** (drugs, metabolites, [cofactors](https://en.wikipedia.org/wiki/Cofactor_(biochemistry)))
- **Metal ions** (Zn, Fe, Mg, Ca, etc.)
- **Nucleic acids** (DNA and RNA)
- **Non-standard residues** (modified amino acids, [post-translational modifications](https://en.wikipedia.org/wiki/Post-translational_modification))

This context is critical for designing functional enzymes, metalloprotein binding sites, and protein-nucleic acid interfaces, where the sequence must be compatible with both the protein fold and its molecular partners.

## Available tools

In [2]:
display_available_tools("ligandmpnn")

- **`run_ligandmpnn_sample()`** — Sample protein sequences using LigandMPNN
- **`run_ligandmpnn_score()`** — Score protein sequences using LigandMPNN

### `run_ligandmpnn_sample`

LigandMPNN analyzes the backbone coordinates and any ligand atoms present in the structure to generate sequences optimized for the target fold and binding environment. We use GFP as a minimal runnable example. For ligand-aware design, provide a structure file that includes HETATM records for the bound ligand of interest. You can optionally fix residue positions to preserve critical interactions and exclude specific amino acids from the designed sequences.

In [3]:
from proto_tools import (
    InverseFoldingStructureInput,
    LigandMPNNSampleConfig,
    LigandMPNNSampleInput,
    LigandMPNNScoringConfig,
    LigandMPNNScoringInput,
    SequenceStructurePair,
    run_ligandmpnn_sample,
    run_ligandmpnn_score,
)
from proto_tools.entities.structures.examples import get_gfp_structure


In [4]:
# Display input docs
display_api_reference("ligandmpnn", "input", "run_ligandmpnn_sample")

# Load GFP structure and configure design constraints
gfp_structure = get_gfp_structure()
struct_input = InverseFoldingStructureInput(
    structure=gfp_structure,
    fixed_positions={"A": [2, 3, 4]},
)
inputs = LigandMPNNSampleInput(inputs=[struct_input])

**Input** — `InverseFoldingInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `inputs` | `list[proto_tools.tools.inverse_folding.shared_data_models.InverseFoldingStructureInput]` | required | Per-structure inputs, each containing a structure and optional selections. |

In [5]:
# Display config docs
display_api_reference("ligandmpnn", "config", "run_ligandmpnn_sample")

# Configure sampling | see docs above for all fields
config = LigandMPNNSampleConfig(
    num_sequences_per_structure=2,  # Generate 2 sequence designs
    temperature=0.1,                # Conservative designs
    excluded_amino_acids=["C"],     # Exclude cysteine
    seed=42,
    device="cuda",                  # Change to "cpu" if no GPU available
)

**Config** — `LigandMPNNSampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on. Options include 'cuda' (NVIDIA GPU), 'cpu' (CPU execution) |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int | None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `num_sequences_per_structure` | `int` | `1` | Total number of sequences to generate per input structure. |
| `batch_size` | `int | None` | `None` | Number of sequences to process simultaneously on GPU. Defaults to num_sequences_per_structure. |
| `temperature` | `float` | `0.1` | Sampling temperature; lower = greedier, higher = more diverse |
| `model_type` | `Literal['ligand_mpnn', 'per_residue_label_membrane_mpnn', 'global_label_membrane_mpnn']` | `'ligand_mpnn'` | LigandMPNN variant: ligand-aware or membrane (per-residue/global) |
| `ligand_mpnn_use_atom_context` | `bool` | `True` | Encode ligand atom context in the message-passing graph |
| `ligand_mpnn_use_side_chain_context` | `bool` | `False` | Condition on sidechain atoms of fixed residues |
| `ligand_mpnn_cutoff_for_score` | `float` | `8.0` | Ligand-residue distance cutoff (A) for interface recovery score |
| `excluded_amino_acids` | `list[Literal['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']] | None` | `None` | Single-letter codes of amino acids to exclude (e.g. ['C'] to forbid cysteine) |

In [6]:
# Run the sampling function
result = run_ligandmpnn_sample(inputs, config)

LigandMPNN sampling:   0%|          | 0/1 [00:00<?, ?structure/s]

In [7]:
# Display output docs
display_api_reference("ligandmpnn", "output", "run_ligandmpnn_sample")

# Print designed sequences
for i, seq_res in enumerate(result.designed_sequences):
    print(f"Structure {i} designs:")
    for j, seq in enumerate(seq_res.sequences, 1):
        preview = f"{seq[:50]}..." if len(seq) > 50 else seq
        print(f"  Design {j}: {preview}")
        print(f"            Length: {len(seq)} residues")


**Output** — `InverseFoldingOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `designed_sequences` | `list[Annotated[proto_tools.tools.inverse_folding.shared_data_models.DesignedSequences, SerializeAsAny()]]` | required | List of sequences predicted for the input structures |

Structure 0 designs:
  Design 1: SKGAELMKGEVPVKVTLEGDVNGKKFKVEGEGHVRAQEGLLELHFECTSG...
            Length: 227 residues
  Design 2: SKGAELLKGRVPVRVHLEGDVNGKKFKVEGEGHGDASRGLLRLHFRCTSG...
            Length: 227 residues


### `run_ligandmpnn_score`

Use `run_ligandmpnn_score` to compute LigandMPNN log-likelihood and perplexity for a sequence in a structure context. This example loads a small protein-only backbone (the in-tree `example_input_fixture.pdb`) and scores its native chain-A sequence — LigandMPNN's score path requires a fully-canonical polymer chain, so chromophore / non-standard residues stay on the sampling side demonstrated above.


In [8]:
display_api_reference("ligandmpnn", "input", "run_ligandmpnn_score")
display_api_reference("ligandmpnn", "config", "run_ligandmpnn_score")
display_api_reference("ligandmpnn", "output", "run_ligandmpnn_score")

# Resolve the in-tree clean-protein fixture by walking up to the repo root.
from pathlib import Path
from proto_tools.entities.structures import Structure
_repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
score_structure = Structure.from_file(str(_repo_root / "proto_tools" / "tools" / "inverse_folding" / "example_input_fixture.pdb"))
score_sequence = score_structure.get_chain_sequence("A", remove_non_standard=True)

score_inputs = LigandMPNNScoringInput(
    sequence_structure_pairs=[SequenceStructurePair(sequence=score_sequence, structure=score_structure)]
)
score_result = run_ligandmpnn_score(
    score_inputs,
    LigandMPNNScoringConfig(seed=42, return_logits=False),
)

score = score_result.scores[0]
print(f"Perplexity: {score.perplexity:.3f}")
print(f"Average log-likelihood: {score.avg_log_likelihood:.3f}")

**Input** — `LigandMPNNScoringInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequence_structure_pairs` | `list[proto_tools.tools.inverse_folding.shared_data_models.SequenceStructurePair]` | required | List of sequence-structure pairs to score |

**Config** — `LigandMPNNScoringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on. |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int | None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `fixed_positions` | `dict[str, list[int]] | None` | `None` | Dictionary mapping chain IDs to fixed positions excluded from scoring. |
| `return_logits` | `bool` | `False` | Whether to include per-position logits in the output. |
| `scoring_mode` | `Literal['single_aa', 'autoregressive']` | `'single_aa'` | Use single-position probabilities or one seed-determined autoregressive order. |

**Output** — `InverseFoldingScoringOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `scores` | `list[proto_tools.tools.inverse_folding.shared_data_models.InverseFoldingScoringMetrics]` | required | List of scoring outputs, one per input sequence-structure pair |

Running run_ligandmpnn_score [00:00]

Perplexity: 5.113
Average log-likelihood: -1.632


## Export Results

Designed sequences can be saved to standard file formats for downstream analysis. JSON format is convenient for preserving the full LigandMPNN metrics alongside the sequences, while FASTA format is useful for feeding results into other sequence analysis tools.

In [9]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="ligandmpnn_designs", export_path=output_dir, file_format="json")
score_result.export(name="ligandmpnn_scores", export_path=output_dir, file_format="json")
print(f"Results exported to {output_dir}")

Results exported to example_output
